# Code to preprocess institutional data

In [ ]:
import pandas as pd
from tqdm import tqdm  

from pathlib import Path
import os
import re

from utils.config import find_repo_root

wd = find_repo_root()
os.chdir(wd)

# import local modules
from utils import data_update, data_homogenize

#TODO check name consistency across functions
#TODO ARG creates 110 instead of 75 ?????

## Peru data (SENAMHI)

In [ ]:
# Usage workflow:
file_paths = list(wd.glob('data/SENAMHI_PERU/raw/*.xlsx'))

# Step 1: Process all files (this is working as expected)
metadata, timeseries = data_homogenize.process_excel_files(file_paths)

# Step 2: Clean metadata and timeseries
timeseries_clean = timeseries.groupby(timeseries.columns, axis=1).first() # this is working as expected
timeseries_clean = timeseries_clean[timeseries_clean.index >= '1950-01-01']
metadata_clean = metadata.drop_duplicates(subset=['gauge_name'])

# merge operator info (in a few cases there are many operators per station)
metadata_clean = metadata.groupby('gauge_name').agg({
    col: 'first' if col not in ['operator'] else lambda x: ', '.join(x.unique())
    for col in metadata.columns if col != 'gauge_name'})

metadata_clean = metadata_clean.sort_values(by='gauge_name').reset_index()
metadata_clean['gauge_id'] = metadata_clean['file_path'].str.extract(r'\((\d+)\)')[0]
metadata_clean['gauge_id'] = metadata_clean['gauge_id'].astype(int).apply(lambda x: f'P{x:08d}')
metadata_clean = metadata_clean.set_index('gauge_id')
timeseries_clean.columns = metadata_clean.index

# Step 3: remove stations with problems in gauge location
metadata_clean = metadata_clean[~metadata_clean.gauge_name.isin(["Alpas", "Querococha", "Paron", "Balsa", "Los Cedros", "Chuquicara"])]
timeseries_clean = timeseries_clean[metadata_clean.index]

# Step 4: Save cleaned data and proceed with delineation
timeseries_clean.to_csv('data/SENAMHI_PERU/SENAMHI_daily_1950_2024.csv', index_label='date')
metadata_clean.to_csv('data/SENAMHI_PERU/SENAMHI_metadata.csv')

# data missing in Chancos (Marcara), Colcas (Colcas), Balsa (Santa), Recreta (Santa), 
# Llanganuco (Llanganuco), Los Cedros (Los Cedros), Pachacoto (Pachacoto), Paron (Paron)
# Querococha (Querococha)

## Argentina data (SNHI)

In [ ]:
# Collect all Excel files (mean daily flow; in spanish caudal medio diario)
excel_files = list(wd.glob("data/SNHI_ARG/raw/*.xlsx"))

# Initialize an empty list to store dataframes
combined_df = []

# Loop through each file and append its data
for file in tqdm(excel_files):
    
    # Read the first row to extract the gauge information
    gauge_info = pd.read_excel(file, nrows=1).columns[0]
    
    # Extract the station number using regex
    match = re.search(r'Estacion (\d+)', gauge_info)
    if match:
        station_number = match.group(1)
        gauge_id = f"X{int(station_number):08d}"  # Format as X00000701
    else:
        gauge_id = "UNKNOWN"  
    
    # Read the file, skipping the first row and using the second row as column names
    df = pd.read_excel(file, header=1)
    df = df.rename(columns = {"Caudal Medio Diario [m3/seg]": gauge_id})
    df['Fecha y Hora'] = pd.to_datetime(df['Fecha y Hora'], format='%d/%m/%Y %H:%M', errors='coerce').dt.date    
    df['Fecha y Hora'] = pd.to_datetime(df['Fecha y Hora'])
    df[gauge_id] = pd.to_numeric(df[gauge_id],  errors='coerce')
    df = df.set_index("Fecha y Hora")
    df = df[~df.index.duplicated(keep="first")]
    combined_df.append(df)

combined_df = pd.concat(combined_df, axis = 1).sort_values(by='Fecha y Hora')
combined_df.index.name = "date"
combined_df = combined_df.loc["1950-01-01":"2025-01-01"]
combined_df.to_csv("data/SNHI_ARG/SNHI_daily_1950_2024.csv")

## Chile data (DGA) + PatagoniaMet data (PMET) [Update]

In [ ]:
final_data = data_update.update_camels_cl_data(
    "data/CAMELS_CL/CAMELS_CL_daily_1950_2020.csv",
    "data/CAMELS_CL/DGA_1960_2025.parquet",
    "data/CAMELS_CL/CAMELS_CL_daily_1950_2025.csv"
)

In [ ]:
final_data = data_update.update_pmet_data(
    "data/PMET_OBS/Q_PMETobs_1950_2020_v11d.csv",
    "data/CAMELS_CL/DGA_1960_2025.parquet",
    "data/SNHI_ARG/SNHI_daily_1950_2024.csv",
    "data/PMET_OBS/Q_PMETobs_1950_2025_v11d.csv"
)